# Appendix E: units and churn depth

Reproduce the FTT consensus-rank heatmap for **German Credit, Diabetes, and GMSC**, using **Random and CTE**, K=100, fold 0, model seed 0, and all five stored background runs. This notebook analyzes existing signed SHAP files; it does not train a model or generate explanations.

The helper is `analysis/appendix_e/churn.py`. It enumerates canonical inputs explicitly, validates every stored query and feature against the ordered fold, computes counts, and exports provenance. Run all cells from any directory inside this project using an environment with NumPy, pandas, scikit-learn, SHAP (for unpickling), Matplotlib, and Jupyter. No research environment is modified.

In [ ]:
from pathlib import Path
import os
import sys
import json
import tempfile

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "appendix-e-mpl"))
sys.dont_write_bytecode = True
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / ".shap_multiplicity_root").is_file() and (p / "results_tree").is_dir())
sys.path.insert(0, str(project_root / "analysis"))
from appendix_e.common import set_plot_style, ensure_directories
from appendix_e.churn import load_canonical_runs, summarize, export_tables, make_figure, OUTPUT
import matplotlib.pyplot as plt
from IPython.display import display

set_plot_style()
ensure_directories()
print("Analysis: FTT, K100, three datasets, Random/CTE, five complete seeds.")

## 1. Canonical files and query correspondence

German and Diabetes use whole files `results_tree/{dataset}_ftt_0_0_{method}_s{seed}_sv.pkl`. GMSC uses exactly **c0–c99**, in numeric chunk order, for each method and seed. The alternate GMSC whole files are not a fallback. All 1,020 required SHAP files must exist, and no rows are trimmed.

Every file must contain finite signed attributions, matching query-array dimensions, and the exact feature order. Concatenated query values must equal the current dataset's ordered test fold produced by `StratifiedKFold(5, shuffle=True, random_state=42)`. Existing GMSC display aliases are mapped without changing column order. This establishes feature-value/row-order correspondence, not unique person identity where duplicate feature vectors exist. The known German Random query #187 remains included as in the existing approved analysis.

In [ ]:
runs, input_files, run_validation, dataset_evidence = load_canonical_runs()
assert len(input_files) == 1020
assert len(run_validation) == 30
assert all(values.shape[0] == 5 for values in runs.values())
display(run_validation.groupby(["dataset", "method"], sort=False).agg(
    runs=("background_seed", "count"), files=("n_files", "sum"),
    N=("n_instances", "first"), features=("n_features", "first")))

## 2. Three distinct counting units

For each query, take the mean **absolute** SHAP across its five runs and select consensus features #1/#2/#3. For each selected feature, count its position in every individual run. Each consensus row therefore has **5N observations**, including all ranks 1 through d; no tail is discarded.

The historical J8 convention is retained exactly: `np.argsort(-mean_abs)` for consensus selection and nested `np.argsort` for per-run ranks, with the historical default `quicksort` now stated explicitly. No stable, lexicographic, or random tie policy is substituted. Top-1 uses J4/J7 `np.argmax(abs(SHAP))`, whose maximum ties select the first feature index. The runtime NumPy version and observed tie counts are exported.

- **Any-seed Top-1:** queries whose five Top-1 features are not all equal, divided by **N**.
- **Pairwise Top-1:** disagreeing Top-1 comparisons across all ten unordered run pairs, divided by **10N**.
- **Consensus rank frequency:** counts at each rank for a particular consensus feature, divided by **5N**.

The average used to choose a consensus feature is not a pair-distance average. These raw churn summaries do not use a calibrated threshold. The same five runs both select the consensus feature and supply its observed ranks.

In [ ]:
rank_frequencies, top1_summary, historical_checks, tie_diagnostics = summarize(runs)
assert len(rank_frequencies) == 204
assert len(top1_summary) == 6
assert len(historical_checks) == 12
for (_, _, _), rows in rank_frequencies.groupby(["dataset", "method", "consensus_rank"]):
    assert int(rows["count"].sum()) == int(rows["denominator"].iloc[0])
display(top1_summary[["dataset", "method", "anyseed_count", "anyseed_denominator", "anyseed_percent",
                      "pairwise_count", "pairwise_denominator", "pairwise_percent"]].round(4))
print("All 12 Top-1 results reproduce figures_b/churn_analysis/summary.json within 1e-12 percentage points.")

## 3. Export counts, source hashes, and validation

Machine-readable files are written under `appendix_e/data/`:

- `churn_rank_frequencies.csv`: count, denominator and percentage for all 204 heatmap cells.
- `churn_top1_summary.csv`: both Top-1 summaries with separate numerators and denominators.
- `churn_input_files.csv`: all 1,020 canonical file paths and SHA-256 hashes.
- `churn_run_validation.csv`: all 30 complete runs and query/feature checks.
- `churn_provenance.json` and `churn_validation.json`: definitions, historical code sources, runtime, tie diagnostics, and the 12 historical-summary comparisons.

The source hashes are checked again after analysis. Raw feature values are not exported.

In [ ]:
validation = export_tables(rank_frequencies, top1_summary, input_files, run_validation,
                           dataset_evidence, historical_checks, tie_diagnostics)
assert validation["status"] == "passed"
assert validation["source_hashes_verified_after_read"]
print(json.dumps({key: validation[key] for key in (
    "status", "source_file_count", "run_count", "rank_frequency_rows",
    "query_values_and_order_verified", "feature_names_and_order_verified",
    "all_18_consensus_rows_sum_to_5N", "all_18_consensus_percent_rows_sum_to_100")}, indent=2))

## 4. Consensus-rank heatmap

Each panel shows consensus #1/#2/#3 against **every actual rank**. All panels use `YlOrRd` on the same **0–100%** scale. Numerical annotations are rounded percentages; `<1` denotes positive frequency below 0.5%, and 0 denotes no observations. Exact counts and unrounded percentages remain in the CSV.

A row concentrated at its consensus rank indicates that the selected feature retains that position; dispersion shows how its observed rank varies. This heatmap does not directly depict either Top-1 churn rate. The companion Top-1 table supplies those separately.

Outputs: `figures/appendix_churn_depth.pdf` (vector) and `.png` (visual review).

In [ ]:
figure = make_figure(rank_frequencies)
display(figure)
plt.close(figure)
print("Saved:", (OUTPUT / "figures/appendix_churn_depth.pdf").relative_to(project_root))

## Interpretation and scope

The reproduced FTT Top-1 any-seed rates are German **8.1/10.1%**, Diabetes **11.4/15.2%**, and GMSC **67.7/62.0%** (Random/CTE). Pairwise rates are respectively **4.4/4.9%**, **5.9/7.3%**, and **42.9/32.0%**. Thus GMSC churn reaches the most important feature. Lower Top-1 rates in German/Diabetes do not establish that every Top-3 change is confined to the third/fourth-rank boundary.

These are descriptive summaries of the stored five-run sample. No new independent runs, training, reference calculation, Mallows calibration, or causal decomposition is performed.